## Quantum Machine Learning

Es como ML clásico pero sustituye una de las partes del pipeline.

En este notebook aprenderemos a:
1. **Comparar** machine learning clásico y QML
2. **Añadir capas** y mejorar un modelo de QML

In [ ]:
!pip install qiskit qiskit-aer qiskit-machine-learning qiskit-algorithms matplotlib pylatexenc

Vamos a usar el dataset de iris, preprocesándolo para **clasificación binaria**.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np

dataset = load_iris()
X_full = dataset.data
y_full = dataset.target

#Clasificación binaria: setosa (0) vs versicolor (1) -> usamos petal length y petal width (las más discriminativas)
mask = y_full < 2
X_raw = X_full[mask, 2:4]  #features 2 y 3: petal length, petal width
y_all = y_full[mask]

#Normalizar a [0, 1] para la codificación cuántica
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Dataset: Iris (setosa vs versicolor)")
print("Features: longitud del pétalo, ancho del pétalo")
print(f"Muestras: {len(X_scaled)} | Clase 0 (setosa): {(y_all==0).sum()} | Clase 1 (versicolor): {(y_all==1).sum()}")
print("\nPrimeras 5 muestras (normalizadas):")
print("  petal_len  petal_wid  label")
for i in range(5):
    print(f"  {X_scaled[i,0]:.3f}      {X_scaled[i,1]:.3f}      {y_all[i]}")

#Train/test split estratificado
indices = list(range(len(X_scaled)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y_all)

X = [(float(X_scaled[i, 0]), float(X_scaled[i, 1])) for i in range(len(X_scaled))]
y = y_all.tolist()

print(f"\nTrain: {len(train_idx)} muestras | Test: {len(test_idx)} muestras")

X_train_np = X_scaled[train_idx]
y_train_np = y_all[train_idx]
X_test_np = X_scaled[test_idx]
y_test_np = y_all[test_idx]

---

## Qué hace cada parte del circuito

### 1) `RY(x)` / `RZ(x)` (encoding de datos/feature map)
`RY(x * pi)` o `RZ(x * pi)` convierte cada feature de entrada en una rotacion del qubit.
- Si `x` cambia, cambia el estado cuantico.
- Es el equivalente a "meter los datos" en el modelo.
- Es comun combinar capas `RY` y `RZ` para aumentar expresividad.


### 2) `RY(theta)` (parametro entrenable/ansatz)
`theta` NO viene del dataset: se aprende durante el entrenamiento.
- Igual que un peso en ML clasico.
- El optimizador (ADAM) ajusta `theta` para minimizar la perdida.

### 3) Entrelazamiento (`CX`, `CZ`, ...)
Cuando hay varios qubits, el entrelazamiento permite que los qubits no se comporten de forma independiente.
- Captura interacciones entre features.
- Suele ayudar cuando la frontera de decision es mas compleja o no lineal.
- Hace que un qubit dependa de otro. NO es exactamente un `if` clasico; es una correlacion cuantica entre estados.


### 4) Capas del circuito
Pensad el circuito como una red neuronal por bloques:
1. Capa de encoding: puertas que dependen de `x` (`RY`, `RZ`).
2. Capa variacional: puertas con parametros entrenables (`theta`).
3. Capa de entrelazamiento: conecta qubits (`CX`, `CZ`).
4. Repeticion de bloques: mas expresividad (como mas profundidad en una red).

### 5) Medicion y prediccion
Al medir, obtenemos conteos de `0/1` y estimamos `p(1)`.
Luego aplicamos umbral (por ejemplo, 0.5) para decidir la clase.
- `shots`: numero de mediciones para estimar probabilidades. Es típico tener cientos o miles de `shots` para obtener la tendencia de los valores obtenidos (la computación cuántica es probabilística, no determinista).

Usando **Qiskit Machine Learning** simplificamos mucho este proceso, te abstrae de la lógica de medición y shots.

> **IMPORTANTE**: `RY(x)`/`RZ(x)` codifican informacion, `RY(theta)` aprende, el entrelazamiento mezcla informacion (si hay 2+ qubits), y las capas repetidas aumentan la capacidad del modelo.

### 6) Optimización del circuito: usamos **ADAM** u otro, como en ML clásico, minimizando la pérdida en train, pero en este caso, sobre los parámetros del circuito (`theta`). Esta optimización se hace en un ordenador clásico.


![Circuito QML](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/1_transformers/img/circuito_qml.png)

---

El **clasificador** será un [circuito](https://quantum.cloud.ibm.com/composer?initial=N4IgjghgzgtiBcIDyAFAogOQIoEEDKAsgAQBMAdAAwDcAOgHYCWdAxgDYCuAJgKZE3jdWDAEYBGMk2b9ademABO3AOZEwAbQAsAXRnNFK5pp315ATwAUoogCoiABwYBKVWorG68gF6Wb9py7cZMx9bB2d1QPoYbmh2RQCtIgBaAD4iQ0i6EAAaEDoIaIQQAFU6ABcGMtZuTnSGeWZ2SpAAXyA) cuántico que codificará los datos de entrada y entrenará un parámetro.

Normalmente se divide en 2 partes, encoding/feature map y training/ansatz.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
import numpy as np


def quantum_classifier(): #El mejor angulo theta es el que maximiza la probabilidad de clasificar correctamente los datos
    x = ParameterVector("x", 2)      #Este parámetro se usará para codificar los datos de entrada, de forma automática
    theta = ParameterVector("θ", 1)  #Este parámetro es el que el optimizador ajustará durante el entrenamiento, se buscará el mejor
    
    qc = QuantumCircuit(1)
    
    #ENCODING: codificar los datos en el qubit
    qc.ry(x[0] * np.pi, 0)   #Feature 1 -> rotación RY
    qc.ry(x[1] * np.pi, 0)   #Feature 2 -> rotación RY
    
    #TRAINING: rotación con parámetro entrenable
    qc.ry(theta[0], 0)       #Parámetro que el optimizador ajustará
    
    return qc, x, theta


Comparamos la precisión del modelo cuántico con una regresión logística tradicional.

In [ ]:
#Comparación: Clásico vs Cuántico
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train_np, y_train_np)
c_acc = clf.score(X_test_np, y_test_np)

print(f"Classical test accuracy: {c_acc:.2%}")

In [ ]:
#Comparación: Clásico vs Cuántico
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import ADAM

#Se crea el circuito y se obtienen los parámetros para que el clasificador se los de como input
qc, input_params, weight_params = quantum_classifier()

#Se crea el modelo cuántico con el circuito, parámetros de entrada y parámetros de peso (entrenables)
qnn = SamplerQNN(
    circuit=qc,
    input_params=input_params,
    weight_params=weight_params
)

#Se crea el clasificador
clasificador_cuantico = NeuralNetworkClassifier(
    neural_network=qnn,
    optimizer=ADAM(maxiter=60, lr=0.10) #Número máximo de iteraciones y tasa de aprendizaje para el optimizador ADAM
)

clasificador_cuantico.fit(X_train_np, y_train_np)
q_acc = clasificador_cuantico.score(X_test_np, y_test_np)

print(f"Quantum test accuracy:   {q_acc:.2%}")


Mientras más parámetros de entrada, más qubits necesita el [circuito](https://quantum.cloud.ibm.com/composer?initial=N4IgjghgzgtiBcIDyAFAogOQIoEEDKAsgAQBMAdAAwDcAOgHYCWdAxgDYCuAJgKZE3jdWDAEYBGMk2b9ademABO3AOZEwAbQAsAXRnNFK5pp315ATwAUoogCoiABwYBKVWorG68gF6Wb9py7cZMx9bB2d1QPoYbmh2RQCtIgBaAD4iQ0i6EAAaEDoIaIQQAFU6ABcGMtZuTnSGeWZ2SpAAXyA) o más operaciones sobre el qubit en el que se trabaje.
Dependiendo del número de clases, se puede plantear el clasificador así:

| Caso | Estrategia | Cómo funciona | Complejidad |
|---|---|---|---|
| **2 clases** | **Un solo circuito binario** | El circuito devuelve `0` o `1` (o probabilidad de clase 1 y luego umbral). | Baja |
| **Más de 2 clases** | **OvR (One-vs-Rest)** | Entrenas varios clasificadores binarios: uno por clase (`clase_i` vs resto). | Media (sencillo de implementar) |
| **Más de 2 clases** | **Un único circuito multiclase** | Usas un circuito con más salidas (por ejemplo, varios qubits) y aplicas **softmax** para mapear a probabilidades por clase. | Alta (requiere rediseñar el circuito) |

---

## Ejercicio 1 - Mejorar los resultados de QML
Para mejorar los resultados del clasificador cuántico, existen varias formas:
- Aumentar la cantidad de ejecuciones
- Añadir más capas (ry, rz)
- Añadir entrelazamiento entre qubits (cx, cz)
- Añadir más parámetros entrenables (theta)

Cread una funcion quantum_classifier nueva cambiando el circuito interno y vuelve a ejecutar la celda de la función objetivo, este ahora tendrá:
 - **2 caracteristicas**.
 - **2 parámetros entrenables** ($\theta$).
 - **2 capas**:
    1. Una primera capa de Encoding de las características `x` (con `RY`) y entrenamiento (con `RY`).
    2. Una capa de entrelazamiento (usando `CX` o `CZ`).
    3. Una segunda capa de Encoding (usando `RX`) y entrenamiento (con RY).

In [ ]:
def quantum_classifier(): #El mejor angulo theta es el que maximiza la probabilidad de clasificar correctamente los datos
    x = ParameterVector("x", 2)     
    theta = ParameterVector("θ", 2)  #Se pueden usar varios parámetros entrenables, no solo uno
    
    #========== TODO: Construir el circuito ==========
    #Recuerda aplicar encoding -> entrenamiento -> entrelazamiento -> encoding -> entrenamiento

    qc = QuantumCircuit(...)
    

    return qc, x, theta

---

## Ejercicio 2 - Entender algunas puertas cuánticas

Cambia las puertas a H y X para ver qué hacen.

In [ ]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()  #El simulador para ejecutar circuitos cuánticos

circuit = QuantumCircuit(1)
# ========== TODO: Añadir puerta cuántica ==========

circuit.measure_all() #Fuera de QML, se necesita medir el circuito para obtener resultados (la librería de QML abtrae esto)

circuit.draw("mpl") #Dibujar el circuito para visualizarlo

result = simulator.run(circuit, shots=1024).result() #Se especifican los shots para obtener una distribución de resultados
counts = result.get_counts(circuit)
print(counts)